In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv(r"projects/unsupervised_learning/data/raw/netflix/NFLX.csv")

In [ ]:
print("Missing values:")
print(df.isna().sum())
print("\nDataset Info:")
print(df.info())

# Convert Date to datetime for time-based filtering
df["Date"] = pd.to_datetime(df["Date"])


In [ ]:
df.sort_values("Date", inplace=True)
df.reset_index(drop=True, inplace=True)

# Remove Adj Close (highly correlated with Close)
if "Adj Close" in df.columns:
    del df["Adj Close"]
print("✓ Data sorted by Date and cleaned")

In [ ]:
# Create target: 1 if tomorrow's close > today's close
df["Tomorrow"] = df["Close"].shift(-1)
df["Target"] = (df["Tomorrow"] > df["Close"]).astype(int)

print(f"Years available: {sorted(df['Date'].dt.year.unique())}")
print(f"Target distribution:\n{df['Target'].value_counts()}")

del df["Tomorrow"]

# Train-test split (2018-2021 train, 2022 test)
x_train = df.loc[df["Date"].dt.year <= 2021].drop(columns=["Target", "Date"])
y_train = df.loc[df["Date"].dt.year <= 2021, "Target"]

x_test = df.loc[df["Date"].dt.year == 2022].drop(columns=["Target", "Date"])
y_test = df.loc[df["Date"].dt.year == 2022, "Target"]

print(f"\nTraining samples: {len(x_train)}, Test samples: {len(x_test)}")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Train model
model = RandomForestClassifier(
    n_estimators=1000,
    random_state=42
)

print("Training model...")
model.fit(x_train, y_train)
print("✓ Training complete!")


In [ ]:
# Make predictions and check accuracy
preds = model.predict(x_test)
accuracy = accuracy_score(preds, y_test)

print(f"Model Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
